In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
import sys
sys.path.append("..")

val_split_date = '2023-06-01'
test_split_date = '2024-01-01'    
seq_length = 30        
forecast_length = 30
epochs = 150
shift_crude_days = 0
warmup_days=14
warmup_epochs=2
warmup_every_n_days=1
act_fn = 'relu'
loss_fn = 'asymmetric_mse'

root = os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'Data')

In [ ]:
with open(os.path.join(root, 'FFNN Hyperparameters.txt'), "r") as f:
    HYPERPARAMETERS = json.load(f)

CONFIGS= {
    "sales_only_no_scaled_no_calender": {
        "use_scaling": False,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    "sales_only_no_scaled_calender_numbers": {
        "use_scaling": False,
        "use_crude": False,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    "sales_only_no_scaled_calender_sincos": {
        "use_scaling": False,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },
    "sales_only_scaled_no_calender": {
        "use_scaling": True,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    "sales_only_scaled_calender_numbers": {
        "use_scaling": True,
        "use_crude": False,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    "sales_only_scaled_calender_sincos": {
        "use_scaling": True,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },

    # -------------------- SALES + CRUDE --------------------
    "sales_and_crude_no_scaled_no_calender": {
        "use_scaling": False,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    "sales_and_crude_no_scaled_calender_numbers": {
        "use_scaling": False,
        "use_crude": True,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    "sales_and_crude_no_scaled_calender_sincos": {
        "use_scaling": False,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },
    "sales_and_crude_scaled_no_calender": {
        "use_scaling": True,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    "sales_and_crude_scaled_calender_numbers": {
        "use_scaling": True,
        "use_crude": True,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    "sales_and_crude_scaled_calender_sincos": {
        "use_scaling": True,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },
}

CONFIG_NAMES = {
    0: "sales_only_no_scaled_no_calender",
    1: "sales_only_no_scaled_calender_numbers",
    2: "sales_only_no_scaled_calender_sincos",
    3: "sales_only_scaled_no_calender",
    4: "sales_only_scaled_calender_numbers",
    5: "sales_only_scaled_calender_sincos",
    6: "sales_and_crude_no_scaled_no_calender",
    7: "sales_and_crude_no_scaled_calender_numbers",
    8: "sales_and_crude_no_scaled_calender_sincos",
    9: "sales_and_crude_scaled_no_calender",
    10: "sales_and_crude_scaled_calender_numbers",
    11: "sales_and_crude_scaled_calender_sincos",
}

In [4]:
cfg = CONFIG_NAMES[10]
num_layers = HYPERPARAMETERS[cfg]['num_layers']
hidden_layers = HYPERPARAMETERS[cfg]['hidden_layers']
dropout = HYPERPARAMETERS[cfg]['dropout']
lr = HYPERPARAMETERS[cfg]['lr']
batch_size = HYPERPARAMETERS[cfg]['batch_size']
under_parameter = HYPERPARAMETERS[cfg]['under_parameter']
over_parameter = HYPERPARAMETERS[cfg]['over_parameter']
sales_scaler = MinMaxScaler() if CONFIGS[cfg]["use_scaling"] else None
crude_scaler = MinMaxScaler() if CONFIGS[cfg]["use_scaling"] else None
use_crude=CONFIGS[cfg]["use_crude"]
use_month=CONFIGS[cfg]["use_month"]
use_month_sin_cos=CONFIGS[cfg]["use_month_sin_cos"]
use_dayofweek=CONFIGS[cfg]["use_dayofweek"]
use_dayofweek_sin_cos=CONFIGS[cfg]["use_dayofweek_sin_cos"]


In [ ]:
from utils import load_and_prepare_data
denoised_sales_df = pd.read_excel(os.path.join(root, 'FixedData_Random_Smoothed.xlsx'))
noised_sales_df = pd.read_excel(os.path.join(root, 'DataMissingValuesFilled.xlsx'))
holiday_df = pd.read_csv(os.path.join(root, 'holidays.csv'))
crude_df = pd.read_csv(os.path.join(root, 'Crude Oil Prices.csv'))

# Load and prepare data
print("Loading data...")

train_dataset, val_dataset, test_dataset = load_and_prepare_data(
                                denoised_sales_df=denoised_sales_df,
                                noised_sales_df=noised_sales_df,
                                crude_df=crude_df,
                                val_split_date=val_split_date,
                                test_split_date=test_split_date,
                                seq_length=seq_length,
                                forecast_length=forecast_length,
                                batch_size=batch_size,
                                sales_scaler=sales_scaler,
                                crude_scaler=crude_scaler,
                                use_crude=use_crude,
                                use_month=use_month,
                                use_month_sin_cos=use_month_sin_cos,
                                use_dayofweek=use_dayofweek,
                                use_dayofweek_sin_cos=use_dayofweek_sin_cos,
                                shift_crude_days=shift_crude_days
                            )

train_loader = train_dataset.get_dataloader(batch_size=batch_size, shuffle=True)
val_loader = val_dataset.get_dataloader(batch_size=batch_size, shuffle=False)


Loading data...
Total records : 1736
Train records : 1156
Val records   : 214
Test records  : 366

Val starts  at: 2023-06-01
Test starts at: 2024-01-01
Using features: ['Sales', 'Crude', 'Month', 'dow']


/Users/milanroy/Desktop/CDIS-Project/ATGL/CodeBase/FFNN/../utils.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])


In [ ]:
from Models import FFNN
from engine import  train_model, generate_predictions, generate_warm_up_predictions
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")
input_dim = train_dataset.X.shape[1] * train_dataset.X.shape[2]

model = FFNN(
            input_size=input_dim,
            output_size=forecast_length,
            hidden_layers=hidden_layers,
            activation=act_fn,
            dropout=dropout
        )

print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

# Train model
print("Training model...")
train_losses, test_losses, optimizer, criterion = train_model(
                            model=model,
                            train_loader=train_loader,
                            val_loader=val_loader,
                            epochs=epochs,
                            lr=lr,
                            device=device,
                            under_parameter=under_parameter,
                            over_parameter=over_parameter,
                            patience=50,
                            loss_fn= loss_fn
                        )


Y_pred_uncorrected, Y_true, Y_true_noised = generate_predictions(model, test_dataset, device=device)



Using device: cpu

Model Parameters: 29,022

Training model...
Epoch 10/150 | Train Loss: 0.006377 | Val Loss: 0.002418
Epoch 20/150 | Train Loss: 0.005592 | Val Loss: 0.002220
Epoch 30/150 | Train Loss: 0.005242 | Val Loss: 0.002446
Epoch 40/150 | Train Loss: 0.004507 | Val Loss: 0.001913
Epoch 50/150 | Train Loss: 0.004664 | Val Loss: 0.002266
Epoch 60/150 | Train Loss: 0.003818 | Val Loss: 0.002389
Epoch 70/150 | Train Loss: 0.004680 | Val Loss: 0.002572
Epoch 80/150 | Train Loss: 0.003553 | Val Loss: 0.002388

Early stopping triggered at epoch 88. Best Val Loss: 0.001929


In [ ]:
from holidayCorrection import holiday_correction
from utils import get_error_df, plot_error_df

Y_pred_corrected = holiday_correction(Y_pred_uncorrected, test_dataset.sample_dates, noised_sales_df, holiday_df)
error_df = get_error_df(Y_true_noised=Y_true_noised, 
                        Y_pred_corrected=Y_pred_corrected, 
                        Y_pred_uncorrected=Y_pred_uncorrected,
                        sample_dates=test_dataset.sample_dates)


plot_error_df(error_df)

/Users/milanroy/Desktop/CDIS-Project/ATGL/CodeBase/FFNN/../utils.py:263: RuntimeWarning: Mean of empty slice
  over_mean  = np.nanmean(over_errors, axis=1)


/Users/milanroy/Desktop/CDIS-Project/.venv/lib/python3.9/site-packages/plotly/graph_objs/_deprecations.py:378: DeprecationWarning:

plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.




Over-prediction >= 10%: 88 days (26.27%)
Under-prediction <= -5%: 137 days (40.90%)
Both conditions: 39 days (11.64%)
Either condition: 186 days (55.52%)


In [17]:
import plotly.graph_objects as go
def plot_one_day(error_df, Y_pred_corrected, Y_pred_uncorrected, Y_true_noised, test_dataset,date):
    
    df = error_df.copy()
    start_date = test_dataset.sample_dates[0]
    Y_pred_corrected = Y_pred_corrected.copy()
    Y_pred_uncorrected = Y_pred_uncorrected.copy()
    Y_true_noised = Y_true_noised.copy()

    i = (pd.to_datetime(date) - pd.to_datetime(start_date)).days
    x = pd.date_range(start=pd.to_datetime(date) + pd.Timedelta(days=1), periods=30)
    fig = go.Figure()
    uncorrected_mape = df["UncorrectedMape"][date]
    corrected_mape = df["CorrectedMape"][date]

    fig.add_trace(go.Scatter(x=x,
                            y=Y_pred_corrected[i],
                            mode='lines+markers',
                            name='Prediction',
                            legendgroup='lines',
                            legend='legend'))
    fig.add_trace(go.Scatter(x=x,
                            y=Y_pred_uncorrected[i],
                            mode='lines+markers',
                            name='Uncorrected Prediction',
                            legendgroup='lines',
                            legend='legend'))
    fig.add_trace(go.Scatter(x=x,
                            y=Y_true_noised[i],
                            mode='lines+markers',
                            name='True values',
                            legendgroup='lines',
                            legend='legend'))
    fig.add_trace(go.Scatter(x=[None], y=[None],
                            mode='markers',
                            marker=dict(opacity=0),
                            showlegend=True,
                            legendgroup='mape',
                            legend='legend2',
                            name=f'Uncorrected MAPE: {uncorrected_mape:.3f}<br>Corrected MAPE:   {corrected_mape:.3f}'))
    fig.update_layout(
        title=f'Prediction at {date}',
        legend=dict(
            title='Legend',
            x=0.01,
            y=0.99,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='black',
            borderwidth=1,
        ),
        legend2=dict(
            title='MAPE',
            x=0.99,
            y=0.99,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='black',
            borderwidth=1,
        )
    )
    fig.show()

plot_one_day(error_df=error_df,
             Y_pred_corrected=Y_pred_corrected,
             Y_pred_uncorrected=Y_pred_uncorrected,
                Y_true_noised=Y_true_noised,
                test_dataset=test_dataset,
                date='2024-10-29')

In [11]:
from utils import plot_one_day
plot_one_day(error_df=error_df,
             Y_pred_corrected=Y_pred_corrected,
             Y_pred_uncorrected=Y_pred_uncorrected,
                Y_true_noised=Y_true_noised,
                test_dataset=test_dataset,
                date='2024-10-30')


In [9]:
from utils import plot_monthly_error_distribution
plot_monthly_error_distribution(error_df=error_df)

In [10]:
from utils import tabulate_error_distribution
tabulate_error_distribution(error_df=error_df)


Daily Corrected MAPE Distribution
+---------+---------+--------------+--------------------+----------------+
| Range   |   Count | Percentage   |   Cumulative Count | Cumulative %   |
+=========+=========+==============+====================+================+
| <3%     |       0 | 0.00%        |                  0 | 0.00%          |
+---------+---------+--------------+--------------------+----------------+
| 3-5%    |      97 | 28.96%       |                 97 | 28.96%         |
+---------+---------+--------------+--------------------+----------------+
| 5-7%    |     101 | 30.15%       |                198 | 59.10%         |
+---------+---------+--------------+--------------------+----------------+
| 7-10%   |      81 | 24.18%       |                279 | 83.28%         |
+---------+---------+--------------+--------------------+----------------+
| 10-15%  |      56 | 16.72%       |                335 | 100.00%        |
+---------+---------+--------------+--------------------+--------